# RAE Layerwise Playground

这个 notebook 只做 RAE-DINOv2 / RAE-MAE / RAE-SigLIP2 的 layerwise 可视化。默认看四个层：位置编码前 `patch_pre_pos`、位置编码后 `post_pos`、中层 `hidden_6`、最终 `rae_normalized`。接口尽量保持简单：`pick_x` 选图，`E_layer` 取层，`P` 做几何变换，`V` / `V_layers` 画图，`M_layers` 批量算每层指标。

In [ ]:
from pathlib import Path
import math
import sys
import importlib

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "train_eqvae").exists() else CWD.parent
sys.path.insert(0, str(ROOT))

from baselines.visual_adapters import RAE_SPECS, get_rae_status, load_rae_adapter
import baselines.dinov2_token_diagnostics as diag

diag = importlib.reload(diag)
P = diag.P
TRANSFORMS = diag.TRANSFORMS
extract_vit_stage_latents = diag.extract_vit_stage_latents
load_named_dataset = diag.load_named_dataset
pick_dataset_images = diag.pick_dataset_images
relative_token_error = diag.relative_token_error

print(f"ROOT = {ROOT}")
print(f"CUDA = {torch.cuda.is_available()}, GPUs = {torch.cuda.device_count()}")

## 1. 配置

In [ ]:
# 数据集。默认用本机 /data/shared 下的 Caltech101。
dataset_root = "/data/shared"
dataset_name = "caltech101"
dataset_split = "train"
dataset_path = ""       # dataset_name="image_folder" 时使用
download_dataset = False
image_size = 256

# RAE 设置。
rae_repo_path = str(ROOT / "external/RAE")
rae_auto_clone = False
rae_auto_download = False
device_name = "cuda:0"
seed = 142

# 最常用的四个 layerwise 位置。
model_key = "rae_dinov2"       # "rae_dinov2", "rae_mae", "rae_siglip2"
layers = ("patch_pre_pos", "post_pos", "hidden_6", "rae_normalized")
hidden_indices = (0, 1, 3, 6, 9, 12)
transform = "rot90"            # 也可用 "flip_h", "flip_v", "translate_right", "translate_down", "zoom_in", "zoom_out"
metric_models = ("rae_dinov2", "rae_mae", "rae_siglip2")
metric_transforms = ("rot90", "rot180", "rot270", "flip_h", "flip_v", "translate_right", "translate_down", "zoom_in", "zoom_out")
metric_count = 128
metric_batch_size = 8

# 可视化设置。
num_images = 4
sample_index = 0
cell_size = 3.2
center = "sample"

## 2. 简洁接口

In [ ]:
device = torch.device(device_name if torch.cuda.is_available() or device_name == "cpu" else "cpu")
dataset = load_named_dataset(dataset_name, dataset_root, dataset_split, download=download_dataset, dataset_path=dataset_path)
RAE_CACHE = {}
SUPPORTED_RAE_KEYS = ("rae_dinov2", "rae_mae", "rae_siglip2")


def check_rae():
    status = get_rae_status(rae_repo_path)
    for key, info in status.items():
        ready = info["repo"] and info["config"] and info["weights"]
        note = "ready" if ready else info
        print(f"{key:12s} {'ready' if ready else 'missing'} {note}")
    return status


def get_rae(key=model_key):
    if key not in RAE_CACHE:
        if key not in SUPPORTED_RAE_KEYS:
            raise ValueError("layerwise notebook 当前只支持 rae_dinov2 / rae_mae / rae_siglip2。")
        RAE_CACHE[key] = load_rae_adapter(
            key,
            repo_path=rae_repo_path,
            device=device,
            dtype=torch.float32,
            auto_clone=rae_auto_clone,
            auto_download=rae_auto_download,
        )
    return RAE_CACHE[key]


def pick_x(index=None, indices=None, count=None, seed=seed, show=True, verbose=True):
    if index is not None:
        indices = [index]
    x, chosen = pick_dataset_images(dataset, count=count or num_images, seed=seed, indices=indices, image_size=image_size)
    x = x.to(device=device, dtype=torch.float32)
    if verbose:
        print("indices", chosen, "x", tuple(x.shape))
    if show:
        display(V(x, title="x"))
    return x


@torch.no_grad()
def E_layers(x, key=model_key, layer_names=layers):
    adapter = get_rae(key)
    stages = extract_vit_stage_latents(adapter, x.to(device=device, dtype=torch.float32), hidden_indices=hidden_indices)
    missing = [name for name in layer_names if name not in stages]
    if missing:
        available = ", ".join(sorted(stages))
        raise KeyError(
            f"缺少 layer {missing}。当前可用 layer: {available}。"
            "如果你刚更新过 notebook，请先重跑第一个导入单元，必要时重启 kernel。"
        )
    return {name: stages[name] for name in layer_names}


def E_layer(x, layer="hidden_6", key=model_key):
    return E_layers(x, key=key, layer_names=(layer,))[layer]


def E_early(x, key=model_key):
    return E_layer(x, "patch_pre_pos", key)


def E_post_pos(x, key=model_key):
    return E_layer(x, "post_pos", key)


def E_mid(x, key=model_key):
    return E_layer(x, "hidden_6", key)


def E_final(x, key=model_key):
    return E_layer(x, "rae_normalized", key)


@torch.no_grad()
def D_final(z, key=model_key):
    return get_rae(key).decode(z.to(device=device, dtype=torch.float32))


def Err(zg, z, g=transform):
    return relative_token_error(zg, P(z, g), center=center).mean().item()


def Err_layer(x, layer="hidden_6", key=model_key, g=transform):
    z = E_layer(x, layer=layer, key=key)
    zg = E_layer(P(x, g), layer=layer, key=key)
    return Err(zg, z, g=g)


def token_pca_image(z, sample=0):
    z0 = z[sample].detach().float().cpu()
    c, h, w = z0.shape
    rows = z0.permute(1, 2, 0).reshape(h * w, c)
    rows = rows - rows.mean(dim=0, keepdim=True)
    try:
        _, _, v = torch.pca_lowrank(rows, q=3, center=False)
        rgb = rows @ v[:, :3]
    except Exception:
        rgb = rows[:, :3]
    rgb = rgb.reshape(h, w, 3)
    rgb = rgb - rgb.amin(dim=(0, 1), keepdim=True)
    rgb = rgb / rgb.amax(dim=(0, 1), keepdim=True).clamp_min(1e-6)
    return rgb.numpy()


def norm_image(z, sample=0):
    m = z[sample].detach().float().pow(2).sum(dim=0).sqrt().cpu()
    m = m - m.min()
    return (m / m.max().clamp_min(1e-6)).numpy()


def image01(x, sample=0):
    img = x[sample].detach().float().cpu().clamp(-1, 1)
    return ((img + 1) * 0.5).permute(1, 2, 0).numpy()


def V(obj, title=None, mode="auto", sample=sample_index, columns=None, cmap="magma"):
    if isinstance(obj, dict):
        items = list(obj.items())
    elif isinstance(obj, (list, tuple)) and obj and isinstance(obj[0], tuple):
        items = list(obj)
    else:
        items = [(title or "obj", obj)]
    columns = columns or min(len(items), 4)
    rows = math.ceil(len(items) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(cell_size * columns, cell_size * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (name, value) in zip(axes, items):
        value = value.detach() if torch.is_tensor(value) else value
        if torch.is_tensor(value) and value.ndim == 4 and value.shape[1] == 3 and mode in {"auto", "image"}:
            ax.imshow(image01(value, sample=sample))
        elif torch.is_tensor(value) and value.ndim == 4:
            ax.imshow(token_pca_image(value, sample=sample) if mode != "norm" else norm_image(value, sample=sample), cmap=None if mode != "norm" else cmap)
        else:
            ax.imshow(value, cmap=cmap)
        ax.set_title(name)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    fig.tight_layout()
    plt.close(fig)
    return fig


def V_layers(x, key=model_key, g=transform, layer_names=layers, sample=sample_index, mode="pca"):
    xg = P(x, g)
    z = E_layers(x, key=key, layer_names=layer_names)
    zg = E_layers(xg, key=key, layer_names=layer_names)
    items = [("x", x), (f"{g}(x)", xg)]
    rows = []
    for name in layer_names:
        err = relative_token_error(zg[name], P(z[name], g), center=center).mean().item()
        rows.append({"model": key, "layer": name, "transform": g, "direct_error": err})
        items.extend([
            (f"{name}: E(x)", z[name]),
            (f"{name}: E({g}x)", zg[name]),
            (f"{name}: P_g E(x)", P(z[name], g)),
        ])
    display(pd.DataFrame(rows))
    return V(items, sample=sample, columns=3, mode=mode)


def V_compare(x, g=transform, layer_names=layers, sample=sample_index):
    xg = P(x, g)
    rows = []
    items = [("x", x), (f"{g}(x)", xg)]
    for key in SUPPORTED_RAE_KEYS:
        z = E_layers(x, key=key, layer_names=layer_names)
        zg = E_layers(xg, key=key, layer_names=layer_names)
        for name in layer_names:
            rows.append({
                "model": key,
                "layer": name,
                "transform": g,
                "direct_error": relative_token_error(zg[name], P(z[name], g), center=center).mean().item(),
            })
            items.append((f"{key}:{name}", z[name]))
    display(pd.DataFrame(rows))
    return V(items, sample=sample, columns=4)


def choose_indices(count=metric_count, seed=seed, indices=None):
    if indices is not None:
        return [int(i) for i in indices]
    count = min(int(count), len(dataset))
    rng = np.random.default_rng(seed)
    return [int(i) for i in rng.permutation(len(dataset))[:count]]


def iter_dataset_batches(indices, batch_size=metric_batch_size):
    indices = [int(i) for i in indices]
    for start in range(0, len(indices), int(batch_size)):
        batch_indices = indices[start:start + int(batch_size)]
        xb, _ = pick_dataset_images(dataset, indices=batch_indices, image_size=image_size)
        yield xb.to(device=device, dtype=torch.float32), batch_indices


@torch.no_grad()
def layerwise_error_detail(keys=metric_models, transforms=metric_transforms, layer_names=layers, count=metric_count, indices=None, seed=seed, batch_size=metric_batch_size, center_mode=center):
    chosen = choose_indices(count=count, seed=seed, indices=indices)
    rows = []
    for key in keys:
        adapter = get_rae(key)
        for xb, batch_indices in iter_dataset_batches(chosen, batch_size=batch_size):
            base = extract_vit_stage_latents(adapter, xb, hidden_indices=hidden_indices)
            valid_layers = [name for name in layer_names if name in base]
            for g in transforms:
                target = extract_vit_stage_latents(adapter, P(xb, g), hidden_indices=hidden_indices)
                for name in valid_layers:
                    per_image = relative_token_error(target[name], P(base[name], g), center=center_mode).detach().cpu().numpy()
                    for dataset_index, err in zip(batch_indices, per_image):
                        rows.append({
                            "model": key,
                            "transform": g,
                            "layer": name,
                            "dataset_index": int(dataset_index),
                            "direct_error": float(err),
                        })
    return pd.DataFrame(rows)


def summarize_layer_errors(detail):
    if detail.empty:
        return pd.DataFrame(columns=["model", "transform", "layer", "mean", "std", "var", "n"])
    summary = (
        detail.groupby(["model", "transform", "layer"], sort=False)["direct_error"]
        .agg(mean="mean", std="std", var="var", n="count")
        .reset_index()
    )
    summary[["std", "var"]] = summary[["std", "var"]].fillna(0.0)
    return summary


def M_layers(keys=metric_models, transforms=metric_transforms, layer_names=layers, count=metric_count, indices=None, seed=seed, batch_size=metric_batch_size, center_mode=center, show=True, return_detail=False):
    detail = layerwise_error_detail(
        keys=keys,
        transforms=transforms,
        layer_names=layer_names,
        count=count,
        indices=indices,
        seed=seed,
        batch_size=batch_size,
        center_mode=center_mode,
    )
    summary = summarize_layer_errors(detail)
    if show:
        display(summary)
    return (summary, detail) if return_detail else summary


def V_metric_summary(
    summary,
    value="mean",
    error="std",
    layer_order=layers,
    model_order=metric_models,
    transform_order=None,
    title=None,
    layout="single",
    panel_width=5.8,
    panel_height=4.2,
):
    if summary.empty:
        raise ValueError("summary 为空，请先运行 M_layers。")
    transform_order = transform_order or [g for g in metric_transforms if g in set(summary["transform"])]
    if not transform_order:
        transform_order = list(dict.fromkeys(summary["transform"].tolist()))
    layer_order = [layer for layer in layer_order if layer in set(summary["layer"])]
    model_order = [model for model in model_order if model in set(summary["model"])]
    if not layer_order or not model_order:
        raise ValueError("summary 里没有匹配的 layer 或 model。")

    colors = {model: plt.cm.tab10(i % 10) for i, model in enumerate(model_order)}
    plot_title = title or f"Layerwise batch direct error ({value} ± {error})"
    short_layer = {
        "patch_pre_pos": "pre",
        "post_pos": "post",
        "hidden_1": "h1",
        "hidden_3": "h3",
        "hidden_6": "h6",
        "hidden_9": "h9",
        "hidden_12": "h12",
        "final_raw": "raw",
        "rae_normalized": "final",
    }

    if layout == "single":
        gap = 0.9
        group_width = len(layer_order)
        width = max(14.0, min(32.0, 2.8 + 0.78 * len(transform_order) * max(1, len(layer_order))))
        fig, ax = plt.subplots(1, 1, figsize=(width, 7.6))
        all_ticks, all_ticklabels = [], []
        legend_handles, legend_labels = [], []

        for group_idx, g in enumerate(transform_order):
            start_x = group_idx * (group_width + gap)
            x = start_x + np.arange(group_width)
            all_ticks.extend(x.tolist())
            all_ticklabels.extend([short_layer.get(layer, layer) for layer in layer_order])
            sub = summary[summary["transform"] == g]
            for model in model_order:
                rows = sub[sub["model"] == model].set_index("layer").reindex(layer_order)
                y = rows[value].to_numpy(dtype=float)
                yerr = rows[error].fillna(0.0).to_numpy(dtype=float) if error in rows else None
                mask = np.isfinite(y)
                if mask.any():
                    handle = ax.errorbar(
                        x[mask],
                        y[mask],
                        yerr=None if yerr is None else yerr[mask],
                        marker="o",
                        capsize=3,
                        linewidth=2.0,
                        markersize=4.5,
                        label=model if group_idx == 0 else None,
                        color=colors[model],
                    )
                    if group_idx == 0:
                        legend_handles.append(handle)
                        legend_labels.append(model)
            center_x = start_x + (group_width - 1) / 2
            ax.text(center_x, -0.20, g, transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=11)
            if group_idx > 0:
                sep = start_x - gap / 2
                ax.axvline(sep, color="0.82", linewidth=1.0)

        ax.set_xticks(all_ticks)
        ax.set_xticklabels(all_ticklabels, rotation=0)
        ax.set_ylabel(value)
        ax.grid(True, axis="y", alpha=0.25)
        ax.set_title(plot_title, pad=56)
        if legend_handles:
            ax.legend(
                legend_handles,
                legend_labels,
                loc="upper center",
                bbox_to_anchor=(0.5, 1.14),
                ncol=len(legend_labels),
                frameon=False,
                borderaxespad=0.0,
            )
        fig.subplots_adjust(left=0.065, right=0.99, top=0.82, bottom=0.24)
        plt.close(fig)
        return fig

    if layout != "grid":
        raise ValueError("layout 只能是 'single' 或 'grid'。")

    ncols = min(3, len(transform_order))
    nrows = math.ceil(len(transform_order) / ncols)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(panel_width * ncols, panel_height * nrows + 1.0),
        sharey=True,
    )
    axes = np.array(axes).reshape(-1)
    x = np.arange(len(layer_order))
    legend_handles, legend_labels = [], []
    for ax_idx, (ax, g) in enumerate(zip(axes, transform_order)):
        sub = summary[summary["transform"] == g]
        for model in model_order:
            rows = sub[sub["model"] == model].set_index("layer").reindex(layer_order)
            y = rows[value].to_numpy(dtype=float)
            yerr = rows[error].fillna(0.0).to_numpy(dtype=float) if error in rows else None
            mask = np.isfinite(y)
            if mask.any():
                handle = ax.errorbar(
                    x[mask],
                    y[mask],
                    yerr=None if yerr is None else yerr[mask],
                    marker="o",
                    capsize=3,
                    linewidth=2.0,
                    markersize=4.5,
                    label=model,
                    color=colors[model],
                )
                if ax_idx == 0:
                    legend_handles.append(handle)
                    legend_labels.append(model)
        ax.set_title(g)
        ax.set_xticks(x)
        ax.set_xticklabels([short_layer.get(layer, layer) for layer in layer_order], rotation=0)
        ax.grid(True, axis="y", alpha=0.25)
        ax.set_ylabel(value)
    for ax in axes[len(transform_order):]:
        ax.axis("off")
    fig.suptitle(plot_title, y=0.985)
    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.935),
            ncol=len(legend_labels),
            frameon=False,
        )
    fig.subplots_adjust(left=0.06, right=0.985, top=0.86, bottom=0.08, hspace=0.40, wspace=0.18)
    plt.close(fig)
    return fig


## 3. 选图、取层、可视化

In [ ]:
# 1. 选图。按需保留一行即可。
x = pick_x(count=num_images, seed=seed)
# x = pick_x(index=123)
# x = pick_x(indices=[0, 5, 9])

# 2. 四个 layerwise latent。变量名尽量短，方便你手动操作。
z0 = E_early(x)   # patch_pre_pos
zp = E_post_pos(x) # post_pos
z1 = E_mid(x)     # hidden_6
z2 = E_final(x)   # rae_normalized

print("z0", tuple(z0.shape), "zp", tuple(zp.shape), "z1", tuple(z1.shape), "z2", tuple(z2.shape))
V({"pre_pos": z0, "post_pos": zp, "mid": z1, "final": z2}, columns=4)

In [ ]:
# 3. 最推荐的单行可视化：看同一模型四个层在一个变换下的情况。
V_layers(x, key=model_key, g=transform, layer_names=layers, sample=sample_index)

In [ ]:
# 4. 对比 DINOv2、MAE 和 SigLIP2 的四个层。
V_compare(x, g="rot90", layer_names=layers, sample=sample_index)
# V_compare(x, g="flip_h", layer_names=layers, sample=sample_index)

In [ ]:
# 5. 你也可以像 latent_playground 一样直接操作 z。
z = E_layer(x, layer="hidden_6", key="rae_dinov2")
zg = E_layer(P(x, "rot90"), layer="hidden_6", key="rae_dinov2")
pz = P(z, "rot90")
err = relative_token_error(zg, pz, center=center).mean().item()
print("hidden_6 rot90 direct error", err)
V([("z", z), ("E(rot90 x)", zg), ("P(z, rot90)", pz)], columns=3)

In [ ]:
# 6. 每一层 direct equivariance error 的批量统计。
# 这个单元不依赖上面选出来的 x；它会从 dataset 中按 seed 抽样，分 batch 计算每层指标。
# 如果想同时比较多个模型或变换，可把 layer_error_models / layer_error_transforms 改成多个值。
layer_error_models = (model_key,)          # 例如 ("rae_dinov2", "rae_mae", "rae_siglip2")
layer_error_transforms = (transform,)      # 例如 ("rot90", "flip_h")
layer_error_count = metric_count
layer_error_batch_size = metric_batch_size
layer_error_hidden_indices = tuple(range(13))
layer_error_include_pos_only = False


def _layer_error_order(stages, hidden_indices=layer_error_hidden_indices, include_pos_only=False):
    order = ["patch_pre_pos"]
    if include_pos_only:
        order.append("pos_only")
    order.append("post_pos")
    order.extend(f"hidden_{i}" for i in hidden_indices)
    order.extend(["final_raw", "rae_normalized"])
    return [name for name in order if name in stages]


def _layer_error_indices(count=layer_error_count, indices=None, seed=seed):
    if indices is not None:
        return [int(i) for i in indices]
    if count <= 0:
        raise ValueError("count 必须大于 0。")
    if len(dataset) < count:
        raise ValueError(f"数据集只有 {len(dataset)} 张，少于请求的 {count} 张。")
    rng = np.random.default_rng(seed)
    return [int(i) for i in rng.permutation(len(dataset))[:count]]


@torch.no_grad()
def M_all_layers(
    keys=layer_error_models,
    transforms=layer_error_transforms,
    hidden_indices=layer_error_hidden_indices,
    count=layer_error_count,
    indices=None,
    seed=seed,
    batch_size=layer_error_batch_size,
    center_mode=center,
    include_pos_only=layer_error_include_pos_only,
    show=True,
    return_detail=False,
):
    chosen = _layer_error_indices(count=count, indices=indices, seed=seed)
    values = {}
    detail_rows = []
    for key in keys:
        adapter = get_rae(key)
        for start in range(0, len(chosen), batch_size):
            batch_indices = chosen[start : start + batch_size]
            xb, _ = pick_dataset_images(dataset, count=len(batch_indices), indices=batch_indices, image_size=image_size)
            xb = xb.to(device=device, dtype=torch.float32)
            base = extract_vit_stage_latents(adapter, xb, hidden_indices=hidden_indices)
            layer_order = _layer_error_order(base, hidden_indices=hidden_indices, include_pos_only=include_pos_only)
            for g in transforms:
                target = extract_vit_stage_latents(adapter, P(xb, g), hidden_indices=hidden_indices)
                for layer in layer_order:
                    per_image = relative_token_error(target[layer], P(base[layer], g), center=center_mode).detach().cpu().numpy()
                    group = (key, g, layer)
                    values.setdefault(group, []).extend(float(v) for v in per_image)
                    if return_detail:
                        for idx, err in zip(batch_indices, per_image):
                            detail_rows.append(
                                {
                                    "model": key,
                                    "transform": g,
                                    "layer": layer,
                                    "index": int(idx),
                                    "direct_error": float(err),
                                }
                            )
    rows = []
    for (key, g, layer), vals in values.items():
        arr = np.asarray(vals, dtype=np.float64)
        rows.append(
            {
                "model": key,
                "transform": g,
                "layer": layer,
                "mean": float(arr.mean()),
                "std": float(arr.std(ddof=0)),
                "var": float(arr.var(ddof=0)),
                "min": float(arr.min()),
                "max": float(arr.max()),
                "n": int(arr.shape[0]),
            }
        )
    summary = pd.DataFrame(rows)
    if show:
        print(f"batch images: {len(chosen)}, batch_size: {batch_size}, center: {center_mode}")
        display(summary)
    if return_detail:
        return summary, pd.DataFrame(detail_rows)
    return summary


def V_all_layer_errors(summary, title=None, value="mean", error="std"):
    if summary.empty:
        raise ValueError("summary 为空，请先运行 M_all_layers。")
    first_group = next(iter(summary.groupby(["model", "transform"], sort=False)))[1]
    layer_order = list(first_group["layer"])
    short = {
        "patch_pre_pos": "pre-pos",
        "pos_only": "pos",
        "post_pos": "post-pos",
        "final_raw": "final raw",
        "rae_normalized": "RAE norm",
    }
    x_pos = np.arange(len(layer_order))
    width = max(13, 0.78 * len(layer_order) + 3)
    fig, ax = plt.subplots(figsize=(width, 5.4))
    for (key, g), sub in summary.groupby(["model", "transform"], sort=False):
        sub = sub.set_index("layer").reindex(layer_order)
        y = sub[value].to_numpy(dtype=float)
        yerr = sub[error].fillna(0.0).to_numpy(dtype=float) if error in sub else None
        ax.errorbar(
            x_pos,
            y,
            yerr=yerr,
            marker="o",
            linewidth=2,
            capsize=3,
            label=f"{key} | {g}",
        )
    ax.set_xticks(x_pos)
    ax.set_xticklabels([short.get(layer, layer.replace("hidden_", "h")) for layer in layer_order], rotation=35, ha="right")
    ax.set_ylabel(f"direct error {value}" + (f" ± {error}" if error else ""))
    ax.set_xlabel("layer")
    ax.set_title(title or "Batch layerwise direct equivariance error")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    plt.close(fig)
    return fig


all_layer_error = M_all_layers(
    keys=layer_error_models,
    transforms=layer_error_transforms,
    count=layer_error_count,
    batch_size=layer_error_batch_size,
)
fig_all_layer_error = V_all_layer_errors(all_layer_error)
fig_all_layer_error


## 5. 批量指标

`M_layers` 会按 `model × transform × layer` 输出 direct error 的均值、标准差、方差和样本数。正式跑大规模时把 `count` 改成 `metric_count`，或直接在配置里把 `metric_count` 设成 512/1024。

In [ ]:
# 快速 smoke：默认只跑少量图片，确认表结构。正式统计用 count=metric_count。
stats_small = M_layers(
    keys=metric_models,
    transforms=("rot90", "flip_h", "translate_right", "zoom_in"),
    layer_names=layers,
    count=metric_count,
    batch_size=4,
)
fig_stats_small = V_metric_summary(stats_small, layout="single")
fig_stats_small
# stats_large = M_layers(count=metric_count, batch_size=metric_batch_size)
# V_metric_summary(stats_large, layout="single")
# V_metric_summary(stats_large, layout="grid")


## 6. ImageNet-1K parquet 与大规模 layerwise 结果

这个 section 使用已经下载到本机的 Hugging Face `ILSVRC/imagenet-1k` parquet 分片，不需要解压成图片目录。默认使用完整的 `test` split 做可视化和 layerwise 统计；当前大规模结果来自 `test` 抽样 2048 张、3 个 RAE encoder、5 个离散变换。


In [ ]:
import json
from IPython.display import Image as IPyImage

imagenet_root = Path("/data/shared/imagenet-1k")
imagenet_manifest_path = imagenet_root / "manifest.json"
imagenet_manifest = json.loads(imagenet_manifest_path.read_text())

split_status = pd.DataFrame(
    [
        {
            "split": split,
            "available_files": info["available_files"],
            "expected_files": info["expected_files"],
            "complete": info["complete"],
            "missing_indices": info["missing_indices"],
        }
        for split, info in imagenet_manifest["splits"].items()
    ]
)
display(split_status)

# 完整 test split：100000 张。这里先抽多一点图片看真实输入质量。
imagenet_test = load_named_dataset(
    "imagenet_parquet",
    "/data/shared",
    split="test",
    dataset_path=str(imagenet_root),
)
print("ImageNet parquet test length:", len(imagenet_test))

x_imagenet, imagenet_indices = pick_dataset_images(
    imagenet_test,
    count=16,
    seed=seed,
    image_size=image_size,
)
x_imagenet = x_imagenet.to(device=device, dtype=torch.float32)
print("indices", imagenet_indices)
V(x_imagenet, title="ImageNet test samples", columns=8)


In [ ]:
# 已跑好的大规模 layerwise 研究结果。
large_layerwise_run = ROOT / "artifacts/layerwise_imagenet/imagenet_test_n2048_3rae_d4"
large_layerwise_summary_path = large_layerwise_run / "summary.csv"
large_layerwise_detail_path = large_layerwise_run / "detail.csv"
large_layerwise_plot_path = large_layerwise_run / "layerwise_summary.png"

large_layerwise_summary = pd.read_csv(large_layerwise_summary_path)
print("summary", large_layerwise_summary.shape, large_layerwise_summary_path)
print("detail", large_layerwise_detail_path)
display(large_layerwise_summary.head(12))

IPyImage(filename=str(large_layerwise_plot_path))


In [ ]:
# 方便快速看最终层/最好层的表。
final_layers = ("patch_pre_pos", "hidden_6", "hidden_12", "final_raw", "rae_normalized")
large_layerwise_key_layers = large_layerwise_summary[
    large_layerwise_summary["layer"].isin(final_layers)
].copy()
display(
    large_layerwise_key_layers
    .pivot_table(index=["model", "transform"], columns="layer", values="mean")
    .reset_index()
)

best_layers = (
    large_layerwise_summary
    .sort_values("mean")
    .groupby(["model", "transform"], as_index=False)
    .first()[["model", "transform", "layer", "mean", "std", "n"]]
)
display(best_layers)

In [ ]:
# 复现实验命令：需要重新跑时，复制下面字符串里的命令到新单元执行。
rerun_layerwise_command = """
CUDA_VISIBLE_DEVICES=0 python ../experiments/rae_layerwise_imagenet_study.py \
  --dataset-split test --count 2048 --batch-size 16 \
  --model-keys rae_dinov2 rae_mae rae_siglip2 \
  --transforms rot90 rot180 rot270 flip_h flip_v \
  --run-name imagenet_test_n2048_3rae_d4
""".strip()
print(rerun_layerwise_command)
